# Import Packages + Set Directory

In [ ]:
#GPT4.1 and Matt Akamatsu, edited + added to by Emma Koves
#uses Zhuanglab storm_analysis package to read .bin file

import argparse
import pathlib
import datetime
import sys
import os
import ot  
import re
import napari
from scipy.stats import pearsonr
from statannotations.Annotator import Annotator
import statsmodels.api as sm
import random
import scipy

import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.stats import mannwhitneyu

from PIL import Image
from skimage import io


''' Select plot settings '''
axes_ticks_size = 14
SMALL_SIZE = 16
MEDIUM_SIZE = 18
BIGGER_SIZE = 20

# plt.rc('font', family='DejaVu Sans')       # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)    # fontsize of the axes title
plt.rc('axes', labelsize=SMALL_SIZE)       # fontsize of the x and y labels
plt.rc('xtick', labelsize=axes_ticks_size)    # fontsize of the tick labels
plt.rc('ytick', labelsize=axes_ticks_size)    # fontsize of the tick labels
# plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title
plt.rcParams["figure.figsize"] = (6, 5.5)

# figure_size = (6, 5.5)  # set figure size manually

# plt.style.use('seaborn-v0_8-colorblind') # set plot style
# sns.set_style('whitegrid')  # set plot style



''' Select directories and paths '''
# Add the network path and parent directory to sys.path
sys.path.append(r"/Users/benjaminbrown/Desktop/ekoves-storm-analysis/storm-analysis/Nas/Microscopy_shared/Meiyan STORM data")
sys.path.append(os.path.abspath(".."))

# Function to load data
def load_with_sa(path, verbose=True):
    """
    Loads an Insight3 .bin file using storm_analysis's readinsight3 module.
    Returns a numpy array of localizations, or None if the file is corrupted.
    """
    from storm_analysis.sa_library import readinsight3
    return readinsight3.loadI3FileNumpy(str(path), verbose=verbose)

# on windows (?) -> it should work for both (except for storm_location...)
# first we define the network path -> idk but doesn't search even if added to sys.path...

#storm_location = r'/Users/benjaminbrown/Desktop/ekoves-storm-analysis/storm-analysis/Nas/Microscopy_shared/Meiyan STORM data'
storm_location = r'/Volumes/ActiveNAS/Microscopy_shared/Charlotte STORM data'

# then the specific path to the storm data

#storm_data_path = r'iPSC-AP2_TagRFP-DNM2-TagGFP-ARPC3-Halo-ActinFix-Phaloidin-AF647-M@Clathrin-CF680'
storm_data_path = r'060518/osmo'

# storm_data_path = r'iPSC-AP2-TagRFP-DNM2-TagGFP-ActinFix-Phaloidin-AF647-M@Clathrin-CF680'


''' Select data to load '''
# just this notebook's convention that left is channel 1, right is channel 2 (doesn't matter, just matches names then)
# left_ch1 = '01-left.bin'
# left_ch1_warp = '01-left_Warp_fwd.bin'
# right_ch2 = '01-right.bin'
localization_file = '20_2c_dc_Warp_fwd.bin'

# # function to extract prefix (assumes format 'NN-' at the start)
# def extract_prefix(filename):
#     match = re.match(r"(\d+)-", filename)
#     if match:
#         return match.group(1)
#     else:
#         match = re.match(r"(\d+)r", filename)
#         if match:
#             return match.group(1)
#         else:
#             raise ValueError(f"Filename {filename} does not start with a numeric prefix followed by '-' or 'r'.")

# prefix = extract_prefix(localization_file)
# loc_file_name = localization_file.replace('.bin', '')  # remove .bin for the name

# If working with multiple files
# # extract prefixes
# prefix_left = extract_prefix(left_ch1)
# prefix_right = extract_prefix(right_ch2)

# # check for prefix mismatch
# if prefix_left != prefix_right:
#     raise ValueError(f"Prefix mismatch: {prefix_left} (left) vs {prefix_right} (right)")
# prefix = prefix_left  # or prefix_right, since they're equal

'''What the notebook should do'''

# save_figures = True
save_figures = False

# save_tiff = True
save_tiff = False     # set to False if you want to skip saving tiff images

run_napari = True       # set to True if you want to run napari
# run_napari = False    # set to False if you want to skip napari

save_napari = True      # set to True if you want to save napari data (such as points selected in the viewer).note that requires run_napari to be True
# save_napari = False   # not recommended

''' Select folder to where the images will be saved'''

# make a figure folder in the parent directory of the notebook
if save_figures:
    parent_dir = os.path.dirname(os.getcwd())  # gets the parent directory of the current working directory
    figures_dir = os.path.join(parent_dir, 'figures', storm_data_path, prefix)
    os.makedirs(figures_dir, exist_ok=True)

# set current date:
if save_figures == True:
    now = datetime.datetime.now()
    date = now.strftime('%Y-%m-%d')

# give an overarching name to figures (such as experiment/project name)
project_name = 'storm_analysis'
# select figure dpi
figure_dpi = 1000

if save_figures:
    print('\nFigures will be saved in the folder: ', figures_dir)
    # print('Figures will have the name in the format:')
    # print("    'project-name_current-date_what-the-figure-is' and have a png ending")
    # print('An example figure name is: cme80_2025-06-01_total_actin_filament_length_histograms_overlaid.png')

    # plt.savefig((figure_folder_path + '/' + project_name + '_' + date + '_' + "total_actin_filament_length_histograms_overlaid.png"),
    #          format='png', bbox_inches='tight', dpi=fig.dpi)
if not save_figures:
    print('\nFigures will not be saved, only shown in the notebook.\n')

In [ ]:
# ---------------------------------------------------------------------------
# Centralised plotting style.
#
# These rcParams define the look of the Figure 1 "Wasserstein distance vs
# time" panels and apply GLOBALLY, so individual plotting cells (and the
# plotting helper) no longer set fonts / ticks / grids / spines. Sizes follow
# the Biophysical Journal / Cell Press spec: Arial (ticks/body 6 pt, axis labels 8 pt bold), 0.6 pt axis/tick
# lines, labelpad 1.0 (previously 12 pt / 1.0 pt line / labelpad 5.0).
#
# Original per-call styling  ->  rcParams equivalent
#   set_x/ylabel(fontsize=12, fontweight='bold', fontfamily='Arial',
#                labelpad=5.0)  -> font.family / axes.labelsize /
#                                  axes.labelweight / axes.labelpad
#   grid(False)                 -> axes.grid = False
#   tick_params(direction='out', length=6, width=1, colors='black',
#               bottom=True, left=True)
#                               -> xtick/ytick.direction / major.size /
#                                  major.width / color + bottom/left
#   spines['top'/'right'].set_visible(False)
#                               -> axes.spines.top / axes.spines.right = False
# ---------------------------------------------------------------------------
import matplotlib as mpl

mpl.rcParams.update({
    # --- Fonts: Arial; body/ticks 6 pt; AXIS LABELS 8 pt BOLD; labelpad 1 (Cell Press / Biophysical J.) ---
    'font.family':      'Arial',
    'font.size':        6,
    'axes.labelsize':   8,
    'axes.labelweight': 'bold',
    'axes.labelpad':    1.0,
    'axes.titlesize':   7,
    'xtick.labelsize':  6,
    'ytick.labelsize':  6,
    'legend.fontsize':  6,

    # --- No grid ---
    'axes.grid':        False,

    # --- Spines / axes lines: hide top & right, 0.6 pt weight ---
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.edgecolor':    'black',
    'axes.linewidth':    0.6,

    # --- Ticks: outward, length 3, width 0.6, black, bottom/left only ---
    'xtick.direction':   'out',
    'ytick.direction':   'out',
    'xtick.major.size':  3,
    'ytick.major.size':  3,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.color':       'black',
    'ytick.color':       'black',
    'xtick.bottom':      True,
    'ytick.left':        True,

    # --- Keep text editable as real fonts in vector exports (Inkscape / Illustrator) ---
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})

# Confirm Arial is actually available -- matplotlib falls back SILENTLY to
# DejaVu Sans otherwise, so a figure can look fine while not being Arial.
from matplotlib import font_manager
try:
    _arial_path = font_manager.findfont('Arial', fallback_to_default=False)
    print(f'Arial resolved to: {_arial_path}')
except Exception:
    print('WARNING: Arial not found -- matplotlib will fall back to its default '
          'sans-serif font. Install Arial or register the .ttf via '
          'font_manager.fontManager.addfont(path).')


In [ ]:
import matplotlib as mpl
print("axes.labelsize:", mpl.rcParams['axes.labelsize'])
print("axes.labelweight:", mpl.rcParams['axes.labelweight'])
print("font.size:", mpl.rcParams['font.size'])
print("xtick.labelsize:", mpl.rcParams['xtick.labelsize'])
print("ytick.labelsize:", mpl.rcParams['ytick.labelsize'])
print("legend.fontsize:", mpl.rcParams['legend.fontsize'])

In [ ]:
# ---------------------------------------------------------------------------
# Output paths -- single source of truth for figure saving.
#
# base_savedir previously lived only in the Figure 1 cell, so the save lines
# raised a NameError if that cell hadn't been run. Defining it once here lets
# every figure cell reference the same location. The per-figure subfolders are
# created up front (guarded) so savefig() won't fail on a missing directory.
# The save calls themselves are unchanged and stay commented until you want them.
# ---------------------------------------------------------------------------
import os

base_savedir = "/Users/admin/Desktop/Abhi/Nonuniformity_manuscript/Panels"#'/Volumes/homes/akamatsuadmin/_Cytosim/abhi_simulations/figures/nonuniformity_manuscript'

for _sub in ('figure1', 'figure2', 'figure3', 'figure4'):
    try:
        os.makedirs(os.path.join(base_savedir, _sub), exist_ok=True)
    except OSError:
        pass  # e.g. network volume not mounted -- ignore until you actually save



def save_panel(fig, name, subdir=None, w_mm=None, h_mm=None):
    """Export a panel at its FINAL physical size, as vector, for 1:1 Inkscape placement.

    The golden rule for font-size compliance: size the panel HERE (matplotlib),
    then drop the SVG into Inkscape at 100% and NEVER rescale it there. Scaling a
    panel down in Inkscape shrinks its text below the 6 pt floor and desyncs it
    from the rcParams spec -- this is exactly what pushed the draft's axis text to
    ~4.3 pt.

      w_mm, h_mm : final printed size in millimetres. Use the target column width
                   (85 = single, 114 = 1.5-col, 174 = full). Omit to keep the
                   figure's current figsize.

    Writes SVG (text stays editable -- svg.fonttype='none') and PDF (dpi only
    affects raster insets; the plot itself is vector). No bbox_inches='tight':
    that trims to content and lets the outer size drift between exports.
    """
    if w_mm is not None and h_mm is not None:
        fig.set_size_inches(w_mm / 25.4, h_mm / 25.4)
    _out = os.path.join(base_savedir, subdir)
    fig.savefig(os.path.join(_out, name + '.svg'), dpi=600, transparent=True)
    fig.savefig(os.path.join(_out, name + '.pdf'), dpi=600, transparent=True)


# Load STORM Data

In [ ]:
base_path = storm_location

# List only subdirectories
subdirs = [name for name in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, name))]

print(subdirs) 

In [ ]:
from pathlib import Path

base_path = Path(storm_location)
folders = ['033017', '060518', '061516']

for f in folders:
    folder_path = base_path / f
    print(f"Checking {folder_path}")
    print("Exists:", folder_path.exists(), "Is dir:", folder_path.is_dir())

    if folder_path.exists() and folder_path.is_dir():
        subdirs = [p for p in folder_path.iterdir() if p.is_dir()]
        for sub in subdirs:
            files = [file.name for file in sub.iterdir() if file.is_file()]
            print(f"Files in {f}/{sub.name}:")
            for file in files:
                print(f"  {file}")
    else:
        print(f"{f} does not exist or is not a directory.")


In [ ]:

storm_location = Path(storm_location)

# Folders to iterate
folders = ['033017', '060518', '061516']

# Dictionary to store DataFrames
all_localizations = {}

for f in folders:
    folder_path = storm_location / f
    if not folder_path.exists():
        print(f"Folder {f} does not exist.")
        continue

    subdirs = [p for p in folder_path.iterdir() if p.is_dir()]
    for sub in subdirs:
        files = [file for file in sub.iterdir() if file.is_file()]
        for file in files:
            try:
                # Load localization
                localizations = load_with_sa(os.path.join(storm_location, f, sub.name, file.name))
                df_localizations = pd.DataFrame(localizations)
                df_localizations['zc'] = df_localizations['zc'] * 0.5 # Scale z-values

                
                # Store in dict
                dict_key = f"{f}/{sub.name}/{file.name}"
                all_localizations[dict_key] = df_localizations
            except Exception as e:
                print(f"Failed to load {file.name}: {e}")

print(f"Loaded {len(all_localizations)} localization files.")


In [ ]:
NWASP_base_path = '/Volumes/ActiveNAS/Microscopy_shared/Charlotte STORM data/SRM_WASP_CLT_2019'

# List only subdirectories
subdirs = [name for name in os.listdir(NWASP_base_path) if os.path.isdir(os.path.join(NWASP_base_path, name))]

print(subdirs)

In [ ]:
from pathlib import Path
import os
import pandas as pd

NWASP_storm_location = Path(r'/Volumes/ActiveNAS/Microscopy_shared/Charlotte STORM data/SRM_WASP_CLT_2019')
folders = ['050319', '060519']

# Dictionary to store DataFrames
NWASP_all_localizations = {}

for f in folders:
    folder_path = NWASP_storm_location / f
    if not folder_path.exists():
        print(f"Folder {f} does not exist.")
        continue

    # List files directly in the folder
    files = [file for file in folder_path.iterdir() if file.is_file()]
    for file in files:
        try:
            # Load localization
            localizations = load_with_sa(file)
            df_localizations = pd.DataFrame(localizations)
            df_localizations['zc'] = df_localizations['zc'] * 0.5  # Scale z-values

            # Store in dict
            dict_key = f"{f}/{file.name}"
            NWASP_all_localizations[dict_key] = df_localizations

        except Exception as e:
            print(f"Failed to load {file.name}: {e}")

print(f"Loaded {len(NWASP_all_localizations)} localization files.")


# Functions

In [ ]:
grid_size = 50

def dbscan_clathrin_segment(gx, gy, df_ch2, eps, min_samples,
                            circular_threshold, r_close, w_clathrin):
    """
    Perform DBSCAN on Clathrin points in a 2D segment (x,y) and filter clusters
    based on circularity, proximity, and width. Keeps full x,y,z coordinates.

    Parameters
    ----------
    gx, gy : int
        Segment coordinates (grid indices)
    df_ch2 : DataFrame
        Clathrin channel points with columns ['xc', 'yc', 'zc']
    eps : float
        DBSCAN eps parameter
    min_samples : int
        DBSCAN min_samples parameter
    circular_threshold : float
        Maximum allowed circularity (λ_max / λ_min)
    r_close : float
        Minimum allowed distance between cluster centers
    w_clathrin : float
        Maximum allowed width of cluster in x or y

    Returns
    -------
    prefiltered_clusters : list of dict
        Each dict: {'label': int, 'points': np.array, 'center': np.array, 'circularity': float}
    final_clusters : list of dict
        Clusters remaining after circularity, width, and proximity filtering
    """
    # --- Segment points ---
    X2 = df_ch2[['xc', 'yc', 'zc']].values  # keep full x,y,z
    xmin, ymin = X2[:,0].min(), X2[:,1].min()
    
    x_bins2 = ((X2[:, 0] - xmin) // grid_size).astype(int)
    y_bins2 = ((X2[:, 1] - ymin) // grid_size).astype(int)
    mask2 = (x_bins2 == gx) & (y_bins2 == gy)
    pts2_all = X2[mask2]  # filtered segment points, with x,y,z

    if len(pts2_all) == 0:
        return [], []  # nothing to cluster in this segment

    # --- DBSCAN in 2D (x,y only) ---
    pts2_2d = pts2_all[:, :2]  # x,y only for clustering
    db2 = DBSCAN(eps=eps, min_samples=min_samples).fit(pts2_2d)
    labels2 = db2.labels_

    prefiltered_clusters = []
    for lbl in set(labels2):
        if lbl == -1:
            continue
        cluster_mask = labels2 == lbl
        cluster_points = pts2_all[cluster_mask]  # full x,y,z points

        if cluster_points.shape[0] < 3:
            continue

        # Covariance + circularity in 2D
        cov = np.cov(cluster_points[:, :2], rowvar=False)
        eigvals = np.sort(np.real(np.linalg.eigvals(cov)))
        circ = np.inf if eigvals[0] <= 0 else eigvals[1] / eigvals[0]

        center = cluster_points.mean(axis=0)  # mean x,y,z

        prefiltered_clusters.append({
            'label': lbl,
            'points': cluster_points,   # full x,y,z
            'center': center,           # mean x,y,z
            'circularity': circ
        })

    # --- Circularity filter ---
    round_clusters = [c for c in prefiltered_clusters if c['circularity'] < circular_threshold]

    # --- Width filter ---
    width_filtered_clusters = []
    for c in round_clusters:
        x_width = c['points'][:,0].max() - c['points'][:,0].min()
        y_width = c['points'][:,1].max() - c['points'][:,1].min()
        if x_width <= w_clathrin and y_width <= w_clathrin:
            width_filtered_clusters.append(c)

    # --- Proximity filter ---
    if len(width_filtered_clusters) == 0:
        final_clusters = []
    else:
        centers = np.array([c['center'] for c in width_filtered_clusters])
        dist_matrix = cdist(centers[:, :2], centers[:, :2])  # distance in x,y only
        too_close = set()
        for i in range(len(width_filtered_clusters)):
            for j in range(i+1, len(width_filtered_clusters)):
                if dist_matrix[i, j] < r_close:
                    too_close.add(i)
                    too_close.add(j)

        final_clusters = [c for idx, c in enumerate(width_filtered_clusters) if idx not in too_close]

    return prefiltered_clusters, final_clusters


In [ ]:

def assign_actin_to_clusters(final_clusters, df_ch1, r, h):
    """
    Assign actin points to Clathrin clusters within a cylindrical region.

    Parameters
    ----------
    final_clusters : list of dict
        Each dict has keys 'label', 'points', 'center'
    df_ch1 : DataFrame
        Actin channel points with columns ['xc', 'yc', 'zc']
    r : float
        Cylinder radius in XY plane
    h : float
        Cylinder height (along Z)

    Returns
    -------
    clusters_with_actin : list of dict
        Each dict contains:
        - 'label': cluster label
        - 'center': cluster center (x,y,z)
        - 'clathrin_points': np.ndarray of clathrin points
        - 'actin_points': np.ndarray of actin points inside cylinder
    """
    all_actin = df_ch1[['xc', 'yc', 'zc']].values
    clusters_with_actin = []

    for c in final_clusters:
        center = np.array(c['center'], dtype=float)
        clathrin_points = np.array(c['points'])

        # --- Cylinder condition ---
        # XY distance from center
        xy_dist = np.linalg.norm(all_actin[:, :2] - center[:2], axis=1)
        # Z within half-height
        z_ok = np.abs(all_actin[:, 2] - center[2]) <= (h / 2.0)

        # Select points that satisfy both
        mask = (xy_dist <= r) & z_ok
        actin_inside = all_actin[mask]

        clusters_with_actin.append({
            'label': c['label'],
            'center': center,
            'clathrin_points': clathrin_points,
            'actin_points': actin_inside
        })

    return clusters_with_actin


In [ ]:
def compute_circular_wd(clusters_with_actin, n_random=10000):
    """
    Compute circular Wasserstein distance for each cluster's Actin points.

    Parameters
    ----------
    clusters_with_actin : list of dict
        Each dict must have keys 'label', 'center', 'clathrin_points', 'actin_points'
    n_random : int
        Number of points for the reference uniform circle

    Returns
    -------
    clusters_with_wd : list of dict
        Same as input, but with additional keys 'wasserstein_distance' and 'corrected_wasserstein_distance'
    """
    clusters_with_wd = []

    # Precompute reference uniform circle
    random_network = np.random.uniform(-np.pi, np.pi, n_random)
    random_network_shift = (np.pi + random_network) / (2*np.pi)  # map [-pi, pi] -> [0,1]

    for c in clusters_with_actin:
        actin_pts = c['actin_points']

        if len(actin_pts) < 2:
            wd = np.nan
            corrected = np.nan
        else:
            # Compute angles of actin points relative to cluster center
            center = c['center']
            directions = np.arctan2(actin_pts[:,1]-center[1], actin_pts[:,0]-center[0])
            directions = (np.pi + directions) / (2*np.pi)  # map [-pi, pi] -> [0,1]

            # WD vs uniform reference
            wd = ot.wasserstein_circle(directions, random_network_shift)

            # Correct against random distributions of same size
            wd_uniforms = []
            for _ in range(10):
                #print(len(directions))
                random_network_2 = np.random.uniform(-np.pi, np.pi, len(directions))
                random_network_2_shift = (np.pi + random_network_2) / (2*np.pi)
                wd_uniforms.append(ot.wasserstein_circle(random_network_2_shift, random_network_shift))

            corrected = wd - np.mean(wd_uniforms)
            #print(corrected)

        # Append WD to cluster info
        cluster_copy = c.copy()
        cluster_copy['wasserstein_distance'] = wd * (2*np.pi)  # scale back to radians
        cluster_copy['corrected_wasserstein_distance'] = corrected
        clusters_with_wd.append(cluster_copy)

    return clusters_with_wd


In [ ]:
def compute_principal_axes(points):
    """
    Compute PCA principal axes for a 3D point cloud.
    
    Returns
    -------
    major : float
        Length along first principal axis
    middle : float
        Length along second principal axis
    minor : float
        Length along third principal axis
    eigvecs : (3,3) array
        Principal axis directions
    """

    if len(points) < 4:
        return np.nan, np.nan, np.nan, None

    pts = np.asarray(points)

    # center
    pts_centered = pts - pts.mean(axis=0)

    # covariance
    cov = np.cov(pts_centered.T)

    # eigen decomposition
    eigvals, eigvecs = np.linalg.eigh(cov)

    # sort largest → smallest
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    # project points onto principal axes
    proj = pts_centered @ eigvecs

    major = proj[:,0].max() - proj[:,0].min()
    middle = proj[:,1].max() - proj[:,1].min()
    minor = proj[:,2].max() - proj[:,2].min()

    return major, middle, minor, eigvecs

In [ ]:
# --- Helper function to select file ---
def select_localization_file(all_localizations):
    print("Available files:")
    for i, key in enumerate(all_localizations.keys()):
        print(f"{i}: {key}")
    idx = int(input("Enter the index of the file to plot: "))
    key = list(all_localizations.keys())[idx]
    return key, all_localizations[key]


In [ ]:
import warnings
def boot_pct_ci(full, other, ci_frac=0.95, reps=10000, seed=0):
    """
    Bootstrap CI of 100*(median(other) - median(full)) / median(full) -- the
    percentage difference in medians relative to the reference. Uses the BCa
    interval (bias- and skew-corrected); if BCa degenerates -- which happens when
    a group has a hard point mass of identical values, e.g. many runs at exactly
    0 monomers -- it falls back to the percentile interval. The ratio is formed
    within each resample, so the interval is bounded >= -100%. `full` must be the
    well-populated reference group (its resampled median is never 0).
    """
    f = np.asarray(full,  float).ravel(); f = f[~np.isnan(f)]
    g = np.asarray(other, float).ravel(); g = g[~np.isnan(g)]

    def _pct(f_s, g_s, axis=-1):
        return 100.0 * (np.median(g_s, axis=axis) - np.median(f_s, axis=axis)) / np.median(f_s, axis=axis)

    def _ci(method):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            r = scipy.stats.bootstrap((f, g), _pct, n_resamples=reps, method=method,
                                      vectorized=True, axis=-1,
                                      confidence_level=ci_frac, random_state=seed)
        return r.confidence_interval.low, r.confidence_interval.high

    lo, hi = _ci('BCa')
    if not (np.isfinite(lo) and np.isfinite(hi)):   # BCa degenerated -> percentile
        lo, hi = _ci('percentile')
    return np.array([lo, hi])

In [ ]:
def _ci_label(df, a, b, column):
    lo, hi = boot_pct_ci(df.loc[df['Distribution']==a, column],
                         df.loc[df['Distribution']==b, column], seed=0)
    return f'95% CI [{lo:.1f}%, {hi:.1f}%]'

In [ ]:
def cv_bootstrap(x, ci_frac=0.95, reps=10000, seed=0):
    """Observed CV (std/mean, ddof=1) and its 95% bootstrap CI. Assumes mean > 0."""
    x = np.asarray(x, float).ravel(); x = x[~np.isnan(x)]
    cv = lambda a, axis=-1: np.std(a, axis=axis, ddof=1) / np.mean(a, axis=axis)
    obs = float(cv(x))
    def _ci(method):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            r = scipy.stats.bootstrap((x,), cv, n_resamples=reps, method=method,
                                      vectorized=True, axis=-1,
                                      confidence_level=ci_frac, random_state=seed)
        return r.confidence_interval.low, r.confidence_interval.high
    lo, hi = _ci('BCa')
    if not (np.isfinite(lo) and np.isfinite(hi)):      # BCa degenerated -> percentile
        lo, hi = _ci('percentile')
    return obs, lo, hi

# Preparing STORM Data (Clustering)

In [ ]:
# Dictionary to store segment_dfs for each file
all_segment_dfs = {}

# Iterate over all loaded localization DataFrames
for key, df_localizations in all_localizations.items():
    print(f"Processing {key}...")
    # Split channels
    df_ch1 = df_localizations[df_localizations['c'] == 1]
    df_ch2 = df_localizations[df_localizations['c'] == 2]
    df_ch9 = df_localizations[df_localizations['c'] == 9]  

    if len(df_ch2) == 0 or len(df_ch1) == 0:
        print(f"Skipping {key}: missing channel 1 or 2")
        continue  # skip if any channel is empty

    # --- Parameters ---
    grid_size = 50  # nm
    r_actin = 1.5
    height = 200

    # Global origin
    xmin = min(df_ch1['xc'].min(), df_ch2['xc'].min())
    ymin = min(df_ch1['yc'].min(), df_ch2['yc'].min())

    # Determine grid ranges
    gx_max = int((max(df_ch1['xc'].max(), df_ch2['xc'].max()) - xmin) // grid_size) + 1
    gy_max = int((max(df_ch1['yc'].max(), df_ch2['yc'].max()) - ymin) // grid_size) + 1

    # Storage for segment DataFrames
    segment_dfs = {}

    for gx in range(1, gx_max - 1):  # skip outermost x-grid
        for gy in range(1, gy_max - 1):  # skip outermost y-grid
            # --- Extract segment points ---
            mask1 = ((df_ch1['xc'] - xmin)//grid_size == gx) & ((df_ch1['yc'] - ymin)//grid_size == gy)
            mask2 = ((df_ch2['xc'] - xmin)//grid_size == gx) & ((df_ch2['yc'] - ymin)//grid_size == gy)
            mask9 = ((df_ch9['xc'] - xmin)//grid_size == gx) & ((df_ch9['yc'] - ymin)//grid_size == gy)

            pts1_seg = df_ch1[['xc','yc']].values[mask1]
            pts2_seg = df_ch2[['xc','yc']].values[mask2]
            pts9_seg = df_ch9[['xc','yc','zc']].values[mask9]

            if len(pts2_seg) == 0:
                continue  # skip empty segments

            # --- Step 1: DBSCAN + cluster filtering ---
            clusters_prefilter, clusters_postfilter = dbscan_clathrin_segment(
                gx, gy, df_ch2, eps=0.2, min_samples=50, circular_threshold=5, r_close=2, w_clathrin=3
            )

            if len(clusters_postfilter) == 0:
                continue  # skip if no good clusters

            # --- Step 2: Assign actin and NWASP points ---
            clusters_with_actin = assign_actin_to_clusters(clusters_postfilter, df_ch1, r=r_actin, h=height)
            clusters_with_nwasp = assign_actin_to_clusters(clusters_postfilter, df_ch9, r=r_actin, h=height)

            # --- Step 3: Compute circular Wasserstein distance ---
            clusters_final_wd = compute_circular_wd(clusters_with_actin)
            clusters_final_wd_nwasp = compute_circular_wd(clusters_with_nwasp)
            #clusters_final_wd_clathrin = compute_circular_wd(clusters_postfilter)

            rows = []
            for c_actin, c_nwasp in zip(clusters_final_wd, clusters_final_wd_nwasp):

                clathrin_pts = np.array(c_actin['clathrin_points'])
                major_axis, middle_axis, minor_axis, eigvecs = compute_principal_axes(clathrin_pts)
                
                rows.append({
                    'label': c_actin['label'],
                    'center': c_actin['center'],
                    'clathrin_points': c_actin['clathrin_points'],
                    'actin_points': c_actin['actin_points'],
                    'wasserstein_distance': c_actin['wasserstein_distance'],
                    'corrected_wasserstein_distance': c_actin['corrected_wasserstein_distance'],
                    'nwasp_points': c_nwasp['actin_points'],
                    'nwasp_wasserstein_distance': c_nwasp['wasserstein_distance'],

                    'major_axis_clathrin': major_axis,
                    'middle_axis_clathrin': middle_axis,
                    'minor_axis_clathrin': minor_axis,

                    'aspect_ratio_clathrin': major_axis / middle_axis if middle_axis > 0 else np.nan,
                    'elongation_clathrin': major_axis / minor_axis if minor_axis > 0 else np.nan,

                    'principal_axes_clathrin': eigvecs
                })

            df_seg = pd.DataFrame(rows)
            segment_dfs[(gx, gy)] = df_seg

    # Save segment_dfs for this file
    all_segment_dfs[key] = segment_dfs

    if len(segment_dfs) == 0:
        print(f"No segments with clusters found for {key}.")
    else:
        print(f"Processed {key}: {len(segment_dfs)} segments.")


In [ ]:

# Initialize dictionaries
filtered_all_segment_dfs = {}
removed_all_segment_dfs = {}
filtered_noiseless_segment_dfs = {}

# Loop over all datasets
for key, segment_dfs in all_segment_dfs.items():
    filtered_segment_dfs = {}
    removed_segment_dfs = {}
    noiseless_segment_dfs = {}
    
    for (gx, gy), df_seg in segment_dfs.items():
        rows_filtered = []
        rows_removed = []
        rows_noiseless = []
        
        for _, c in df_seg.iterrows():
            actin_pts = np.array(c['actin_points'])
            clathrin_pts = np.array(c['clathrin_points'])
            
            # If no actin points at all, mark as removed
            if len(actin_pts) == 0:
                rows_removed.append({
                    'label': c['label'],
                    'center': c['center'],
                    'clathrin_points': clathrin_pts,
                    'actin_points': actin_pts,
                    'wasserstein_distance': c['wasserstein_distance'],
                    'corrected_wasserstein_distance': c['corrected_wasserstein_distance'],
                    'nwasp_points': c['nwasp_points'],
                    'nwasp_wasserstein_distance': c['nwasp_wasserstein_distance']
                })
                continue
            
            # --- DBSCAN on actin points (x,y) ---
            db = DBSCAN(eps=0.2, min_samples=50)
            labels = db.fit_predict(actin_pts[:, :2])
            
            # Kept clusters (any DBSCAN cluster found)
            if np.any(labels != -1):
                rows_filtered.append({
                    'label': c['label'],
                    'center': c['center'],
                    'clathrin_points': clathrin_pts,
                    'actin_points': actin_pts,
                    'wasserstein_distance': c['wasserstein_distance'],
                    'corrected_wasserstein_distance': c['corrected_wasserstein_distance'],
                    'nwasp_points': c['nwasp_points'],
                    'nwasp_wasserstein_distance': c['nwasp_wasserstein_distance']
                })
                
                # Noiseless version: only keep points assigned to cluster
                assigned_pts = actin_pts[labels != -1]
                rows_noiseless.append({
                    'label': c['label'],
                    'center': c['center'],
                    'clathrin_points': clathrin_pts,
                    'actin_points': assigned_pts,
                    'wasserstein_distance': c['wasserstein_distance'],
                    'corrected_wasserstein_distance': c['corrected_wasserstein_distance'],
                    'nwasp_points': c['nwasp_points'],
                    'nwasp_wasserstein_distance': c['nwasp_wasserstein_distance']
                })
            else:
                # All noise → removed
                rows_removed.append({
                    'label': c['label'],
                    'center': c['center'],
                    'clathrin_points': clathrin_pts,
                    'actin_points': actin_pts,
                    'wasserstein_distance': c['wasserstein_distance'],
                    'corrected_wasserstein_distance': c['corrected_wasserstein_distance'],
                    'nwasp_points': c['nwasp_points'],
                    'nwasp_wasserstein_distance': c['nwasp_wasserstein_distance']
                })
        
        # Save per-grid DataFrames if non-empty
        if len(rows_filtered) > 0:
            filtered_segment_dfs[(gx, gy)] = pd.DataFrame(rows_filtered)
        if len(rows_removed) > 0:
            removed_segment_dfs[(gx, gy)] = pd.DataFrame(rows_removed)
        if len(rows_noiseless) > 0:
            noiseless_segment_dfs[(gx, gy)] = pd.DataFrame(rows_noiseless)
    
    # Store in main dictionaries
    filtered_all_segment_dfs[key] = filtered_segment_dfs
    removed_all_segment_dfs[key] = removed_segment_dfs
    filtered_noiseless_segment_dfs[key] = noiseless_segment_dfs
    
    # Print summary
    n_kept = sum(len(df) for df in filtered_segment_dfs.values())
    n_removed = sum(len(df) for df in removed_segment_dfs.values())
    n_noiseless = sum(len(df) for df in noiseless_segment_dfs.values())
    print(f"Filtered {key}, kept {n_kept} clusters, removed {n_removed} clusters, noiseless {n_noiseless} clusters")


# Figure 2

In [ ]:
# --- Select file ---
file_key, df_localizations = select_localization_file(all_localizations)

# --- Split channels ---
df_ch1 = df_localizations[df_localizations['c'] == 1]  # Actin
df_ch2 = df_localizations[df_localizations['c'] == 2]  # Clathrin (all points)

# --- Unit conversion: divide by 10, label as um ---
_unit_divisor = 10  # CHECK: nm -> um is /1000, not /10 -- confirm this is correct for your source units
_unit_label = 'μm'

X1 = df_ch1[['xc', 'yc']].values / _unit_divisor
Z1 = (df_ch1['zc'].values / _unit_divisor) if 'zc' in df_ch1.columns else np.zeros(len(df_ch1))

X2 = df_ch2[['xc', 'yc']].values / _unit_divisor
Z2 = (df_ch2['zc'].values / _unit_divisor) if 'zc' in df_ch2.columns else np.zeros(len(df_ch2))

# --- Biophysical J. / Cell Press panel spec (FIGURE_PREP_PLAN.md), scoped to this
# panel only via rc_context so the rest of the notebook keeps its own style.
# Final size is set directly on the figure (single-column, 82.5 x 82.5 mm square)
# and must not be rescaled afterward -- resize figsize/w/h below instead. ---
_cellpress_style = {
    'font.family':      'Arial',
    'font.size':        6,
    'axes.linewidth':   0.6,
    'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'pdf.fonttype': 42, 'svg.fonttype': 'none',  # fonts stay embedded/editable
}

with mpl.rc_context(_cellpress_style):
    w_mm, h_mm = 70, 70  # single-column square panel (safe target, Sec. 2)
    fig, ax = plt.subplots(figsize=(w_mm / 25.4, h_mm / 25.4))

    # --- Plot Actin (XY) colored by Z ---
    if len(X1) > 0:
        ax.scatter(
            X1[:, 0], X1[:, 1],
            c=Z1, cmap='viridis', s=0.00001, alpha=0.5,
            rasterized=True, label='Actin'
        )

    # --- Plot all Clathrin points (XY), solid purple ---
    if len(X2) > 0:
        ax.scatter(
            X2[:, 0], X2[:, 1],
            c='purple', s=0.00001, alpha=0.5,
            rasterized=True, label='Clathrin'
        )

    # --- True spatial aspect; ticks/spines styled by the rc_context above ---
    ax.set_aspect('equal')
    ax.set_xlabel(f'X ({_unit_label})')
    ax.set_ylabel(f'Y ({_unit_label})')

    # --- Legend: text colored to match each marker, fully opaque, transparent background ---
    legend = ax.legend(loc='upper right', markerscale=50, frameon=False)
    legend.get_frame().set_alpha(0)

    _viridis_mid = plt.cm.viridis(0.5)  # ASSUMPTION: single representative color for the Actin colormap; adjust (e.g. 0.0/1.0) if you want the low/high end instead
    _label_colors = {'Actin (viridis)': _viridis_mid, 'Clathrin': 'purple'}
    for text in legend.get_texts():
        if text.get_text() in _label_colors:
            text.set_color(_label_colors[text.get_text()])
            text.set_alpha(1.0)  # force full opacity regardless of marker alpha

    # --- Scale bar (required on all microscopy panels, Sec. 2) — top-left ---
    all_x = np.concatenate([X1[:, 0], X2[:, 0]]) if (len(X1) or len(X2)) else np.array([0., 1.])
    all_y = np.concatenate([X1[:, 1], X2[:, 1]]) if (len(X1) or len(X2)) else np.array([0., 1.])
    span = max(all_x.max() - all_x.min(), all_y.max() - all_y.min())

    _nice_lengths = np.array([0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50])  # um
    bar_len = _nice_lengths[np.argmin(np.abs(_nice_lengths - 0.2 * span))]

    x0 = all_x.min() + 0.05 * (all_x.max() - all_x.min())
    y0 = all_y.max() - 0.05 * (all_y.max() - all_y.min())  # near top instead of bottom
    ax.plot([x0, x0 + bar_len], [y0, y0], color='black', linewidth=1.0, solid_capstyle='butt')
    ax.text(x0 + bar_len / 2, y0 - 0.03 * (all_y.max() - all_y.min()), f'{bar_len} {_unit_label}',
            ha='center', va='top', fontsize=6, fontfamily='Arial')

    plt.tight_layout()
    # Export at final size, vector, no bbox_inches='tight' (Sec. 1/5) --
    # place the SVG in Inkscape at 100% and never rescale it there.
    save_panel(fig, '2a', subdir='figure2', w_mm=w_mm, h_mm=h_mm)
    #fig.savefig('2a_test.svg', dpi=600, transparent=True)
    #fig.savefig('2a.pdf', dpi=600, transparent=True)
    plt.show()

In [ ]:
# --- Select file ---
file_key, df_localizations = select_localization_file(all_localizations)

# --- Split channels ---
df_ch1 = df_localizations[df_localizations['c'] == 1]  # Actin
df_ch2 = df_localizations[df_localizations['c'] == 2]  # Clathrin (all points)

# --- Unit conversion: divide by 10, label as um ---
_unit_divisor = 10
_unit_label = 'μm'

X1 = df_ch1[['xc', 'yc']].values / _unit_divisor
X2 = df_ch2[['xc', 'yc']].values / _unit_divisor

# No local rc_context / _cellpress_style here -- relies entirely on the
# centralized rcParams block run earlier in the notebook. Only panel size
# is set locally, since that's inherently per-panel, not global style.
w_mm, h_mm = 70, 70
fig, ax = plt.subplots(figsize=(w_mm / 25.4, h_mm / 25.4))

# --- Actin: manual 2D histogram, grayscale, transposed to match imshow/scatter orientation ---
_n_bins = 1200
_clim_max = 20

actin_x_min, actin_x_max = 0., 1.
actin_y_min, actin_y_max = 0., 1.

if len(X1) > 0:
    hist_ch1, xedges_ch1, yedges_ch1 = np.histogram2d(X1[:, 0], X1[:, 1], bins=_n_bins)
    hist_ch1 = hist_ch1.T

    actin_x_min, actin_x_max = xedges_ch1[0], xedges_ch1[-1]
    actin_y_min, actin_y_max = yedges_ch1[0], yedges_ch1[-1]

    extent = [actin_x_min, actin_x_max, actin_y_min, actin_y_max]
    im = ax.imshow(
        hist_ch1, extent=extent, origin='lower',
        cmap='gray', aspect='auto', interpolation='none',
        rasterized=True,
        vmin=0, vmax=_clim_max
    )
    cbar = fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
    cbar.set_label('Count', fontsize=6, fontfamily='Arial')
    cbar.ax.tick_params(labelsize=6, width=0.6)
    cbar.outline.set_linewidth(0.6)

if len(X2) > 0:
    _clathrin_mask = (
        (X2[:, 0] >= actin_x_min) & (X2[:, 0] <= actin_x_max) &
        (X2[:, 1] >= actin_y_min) & (X2[:, 1] <= actin_y_max)
    )
    X2_trimmed = X2[_clathrin_mask]

    if len(X2_trimmed) > 0:
        ax.scatter(
            X2_trimmed[:, 0], X2_trimmed[:, 1],
            c='deeppink', s=0.0045, alpha=0.7,
            rasterized=True, label='Clathrin',
            edgecolors='none', linewidths=0
        )

ax.set_xlim(actin_x_min, actin_x_max)
ax.set_ylim(actin_y_min, actin_y_max)

ax.set_aspect('equal')
ax.set_xlabel(f'X ({_unit_label})')
ax.set_ylabel(f'Y ({_unit_label})')

from matplotlib.lines import Line2D
proxy_actin = Line2D([], [], linestyle='none', marker='none')
proxy_clathrin = Line2D([], [], linestyle='none', marker='none')

legend = ax.legend(
    handles=[proxy_actin, proxy_clathrin],
    labels=['Actin (density)', 'Clathrin'],
    loc='upper left',
    frameon=False,
    handlelength=0,
    handletextpad=0
)
legend.get_frame().set_alpha(0)

_label_colors = {'Actin (density)': '1', 'Clathrin': 'deeppink'}
for handle, text in zip(legend.legend_handles, legend.get_texts()):
    handle.set_visible(False)
    if text.get_text() in _label_colors:
        text.set_color(_label_colors[text.get_text()])
        text.set_alpha(1.0)

span = max(actin_x_max - actin_x_min, actin_y_max - actin_y_min)
_nice_lengths = np.array([0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50])
bar_len = _nice_lengths[np.argmin(np.abs(_nice_lengths - 0.2 * span))]

x0 = actin_x_min + 0.05 * (actin_x_max - actin_x_min)
y0 = actin_y_min + 0.05 * (actin_y_max - actin_y_min)
ax.plot([x0, x0 + bar_len], [y0, y0], color='white', linewidth=1.0, solid_capstyle='butt')
ax.text(x0 + bar_len / 2, y0 + 0.03 * (actin_y_max - actin_y_min), f'{bar_len} {_unit_label}',
        ha='center', va='bottom', color='white', fontsize=6, fontfamily='Arial')

plt.tight_layout()
save_panel(fig, '2a_hist', subdir='figure2', w_mm=w_mm, h_mm=h_mm)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# --- Select dataset ---
dataset_key = "061516/control/12_zc_new3D_Warp_fwd.bin"
seg_dfs = removed_all_segment_dfs[dataset_key]

# --- Collect all clusters ---
all_clusters = []
for df_seg in seg_dfs.values():
    for _, row in df_seg.iterrows():
        wd = row['wasserstein_distance']
        if isinstance(wd, (list, np.ndarray)):
            wd = float(np.squeeze(wd))
        all_clusters.append({
            'site': row['label'],
            'WD': wd,
            'center': np.array(row['center']) if row['center'] is not None else np.zeros(3),
            'actin_points': row['actin_points'],
            'clathrin_points': row['clathrin_points']
        })

df_clusters = pd.DataFrame(all_clusters)

# --- Select clusters ---
n_select = 1
df_selected = df_clusters.sample(n=n_select, random_state=3)  # fixed seed for reproducibility

# --- Unit conversion: divide by 10, label as um ---
_unit_divisor = 10  # CHECK: nm -> um is /1000, not /10 -- confirm this matches your source units
_unit_label = 'μm'

# --- Plot one figure per site ---
for _, cluster in df_selected.iterrows():
    site = cluster['site']
    wd_value = cluster['WD']  # pulled out to avoid quote conflict in the f-string below

    fig, ax = plt.subplots(figsize=(5, 5))

    # Actin XY with Z-coloring -- absolute coordinates, converted units
    actin_pts = np.array(cluster['actin_points'])
    if len(actin_pts) > 0:
        actin_pts_conv = actin_pts / _unit_divisor
        z_actin = actin_pts_conv[:, 2] if actin_pts_conv.shape[1] > 2 else np.zeros(len(actin_pts_conv))
        ax.scatter(
            actin_pts_conv[:, 0], actin_pts_conv[:, 1],
            c='gray', s=1, alpha=0.6,
            label=f'Actin'
        )

    # Clathrin XY -- absolute coordinates, converted units
    clathrin_pts = np.array(cluster['clathrin_points'])
    if len(clathrin_pts) > 0:
        clathrin_pts_conv = clathrin_pts / _unit_divisor
        ax.scatter(
            clathrin_pts_conv[:, 0], clathrin_pts_conv[:, 1],
            c='deeppink', s=1, alpha=0.6, label='Clathrin'
        )

    # --- No manual xticks/yticks/tick_params overrides -- inherits global rcParams ---
    ax.set_xlabel(f'X ({_unit_label})')
    ax.set_ylabel(f'Y ({_unit_label})')
    #ax.legend(loc='upper right', markerscale=3)
    ax.set_aspect('equal')
    ax.set_title("Actin Negative", fontsize=8, fontweight='bold')
    save_panel(fig, f'2a_1', subdir='figure2', w_mm=30, h_mm=30)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# --- Select dataset ---
dataset_key = "061516/control/12_zc_new3D_Warp_fwd.bin"
seg_dfs = filtered_all_segment_dfs[dataset_key]

# --- Collect all clusters ---
all_clusters = []
for df_seg in seg_dfs.values():
    for _, row in df_seg.iterrows():
        wd = row['wasserstein_distance']
        if isinstance(wd, (list, np.ndarray)):
            wd = float(np.squeeze(wd))
        all_clusters.append({
            'site': row['label'],
            'WD': wd,
            'center': np.array(row['center']) if row['center'] is not None else np.zeros(3),
            'actin_points': row['actin_points'],
            'clathrin_points': row['clathrin_points']
        })

df_clusters = pd.DataFrame(all_clusters)

# --- Select clusters ---
n_select = 1
df_selected = df_clusters.sample(n=n_select, random_state=3)  # fixed seed for reproducibility

# --- Unit conversion: divide by 10, label as um ---
_unit_divisor = 10  # CHECK: nm -> um is /1000, not /10 -- confirm this matches your source units
_unit_label = 'μm'

# --- Plot one figure per site ---
for _, cluster in df_selected.iterrows():
    site = cluster['site']
    wd_value = cluster['WD']  # pulled out to avoid quote conflict in the f-string below

    fig, ax = plt.subplots(figsize=(5, 5))

    # Actin XY with Z-coloring -- absolute coordinates, converted units
    actin_pts = np.array(cluster['actin_points'])
    if len(actin_pts) > 0:
        actin_pts_conv = actin_pts / _unit_divisor
        z_actin = actin_pts_conv[:, 2] if actin_pts_conv.shape[1] > 2 else np.zeros(len(actin_pts_conv))
        ax.scatter(
            actin_pts_conv[:, 0], actin_pts_conv[:, 1],
            c='gray', s=1, alpha=0.6,
            label=f'Actin'
        )

    # Clathrin XY -- absolute coordinates, converted units
    clathrin_pts = np.array(cluster['clathrin_points'])
    if len(clathrin_pts) > 0:
        clathrin_pts_conv = clathrin_pts / _unit_divisor
        ax.scatter(
            clathrin_pts_conv[:, 0], clathrin_pts_conv[:, 1],
            c='deeppink', s=1, alpha=0.6, label='Clathrin'
        )

    # --- No manual xticks/yticks/tick_params overrides -- inherits global rcParams ---
    ax.set_xlabel(f'X ({_unit_label})')
    ax.set_ylabel(f'Y ({_unit_label})')
    #ax.legend(loc='upper right', markerscale=3)
    ax.set_aspect('equal')
    ax.set_title("Actin Positive", fontsize=8, fontweight='bold')
    save_panel(fig, f'2a_2', subdir='figure2', w_mm=30, h_mm=30)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Select dataset ---
dataset_key = "061516/control/12_zc_new3D_Warp_fwd.bin"
seg_dfs = filtered_all_segment_dfs[dataset_key]

# --- Collect all clusters ---
all_clusters = []
for df_seg in seg_dfs.values():
    for _, row in df_seg.iterrows():
        wd = row['wasserstein_distance']
        if isinstance(wd, (list, np.ndarray)):
            wd = float(np.squeeze(wd))

        clathrin_pts = np.array(row['clathrin_points']) if len(row['clathrin_points']) > 0 else np.empty((0, 3))

        # --- Compute Clathrin aspect ratio (z/x), absolute coordinates, no centering ---
        if len(clathrin_pts) > 0:
            x_clathrin = clathrin_pts[:, 0]
            z_clathrin = clathrin_pts[:, 2] if clathrin_pts.shape[1] > 2 else np.zeros(len(clathrin_pts))

            x_5, x_95 = np.percentile(x_clathrin, [5, 95])
            z_5, z_95 = np.percentile(z_clathrin, [5, 95])
            width_x = np.abs(x_95 - x_5)
            width_z = np.abs(z_95 - z_5)
            aspect_ratio = width_z / width_x if width_x != 0 else np.nan
        else:
            aspect_ratio = np.nan

        all_clusters.append({
            'site': row['label'],
            'WD': wd,
            'actin_points': row['actin_points'],
            'clathrin_points': row['clathrin_points'],
            'aspect_ratio': aspect_ratio
        })

# --- Filter to valid, low aspect ratio clusters (AR < 200) ---
subset = [c for c in all_clusters if not np.isnan(c['aspect_ratio']) and c['aspect_ratio'] < 200]

random.seed(5)
random_clusters = random.sample(subset, 1)

# --- Unit conversion: divide by 10, label as um ---
_unit_divisor = 10  # CHECK: nm -> um is /1000, not /10 -- confirm this matches your source units
_unit_label = 'μm'

# --- Plot selected site ---
for cluster in random_clusters:
    site = cluster['site']
    wd_value = cluster['WD']  # pulled out to avoid quote conflict in the f-string below
    ar_value = cluster['aspect_ratio']

    fig, ax = plt.subplots(figsize=(30 / 25.4, 30 / 25.4))  # matches save_panel size below

    # Clathrin XZ -- absolute coordinates, converted units, no centering
    clathrin_pts = np.array(cluster['clathrin_points'])
    if len(clathrin_pts) > 0:
        clathrin_pts_conv = clathrin_pts / _unit_divisor
        z_clathrin_conv = clathrin_pts_conv[:, 2] if clathrin_pts_conv.shape[1] > 2 else np.zeros(len(clathrin_pts_conv))
        ax.scatter(
            clathrin_pts_conv[:, 0], z_clathrin_conv,
            c='deeppink', s=1, alpha=0.6, label='Clathrin'
        )

    # --- No manual xticks/yticks/tick_params overrides -- inherits global rcParams ---
    ax.set_ylim(-20,20)
    mid_x = (clathrin_pts_conv[:, 0].min() + clathrin_pts_conv[:, 0].max()) / 2
    ax.set_xlim(mid_x - 0.05, mid_x + 0.05)
    ax.set_xlabel(f'X ({_unit_label})')
    ax.set_ylabel(f'Z ({_unit_label})')
    #ax.legend(loc='upper right', markerscale=3)
    #ax.set_aspect('equal')
    ax.set_title(f"Actin Positive (AR {ar_value:.2f})", fontsize=8, fontweight='bold')
    save_panel(fig, '2a_3', subdir='figure2', w_mm=30, h_mm=30)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Select dataset ---
dataset_key = "061516/control/12_zc_new3D_Warp_fwd.bin"
seg_dfs = filtered_all_segment_dfs[dataset_key]

# --- Collect all clusters ---
all_clusters = []
for df_seg in seg_dfs.values():
    for _, row in df_seg.iterrows():
        wd = row['wasserstein_distance']
        if isinstance(wd, (list, np.ndarray)):
            wd = float(np.squeeze(wd))

        clathrin_pts = np.array(row['clathrin_points']) if len(row['clathrin_points']) > 0 else np.empty((0, 3))

        # --- Compute Clathrin aspect ratio (z/x), absolute coordinates, no centering ---
        # Z gets an extra /100 on top of the standard /10 unit conversion (applied consistently
        # here and in the plotting section below, so the aspect ratio and the plotted shape agree)
        if len(clathrin_pts) > 0:
            x_clathrin = clathrin_pts[:, 0]
            z_clathrin = (clathrin_pts[:, 2] / 100) if clathrin_pts.shape[1] > 2 else np.zeros(len(clathrin_pts))

            x_5, x_95 = np.percentile(x_clathrin, [5, 95])
            z_5, z_95 = np.percentile(z_clathrin, [5, 95])
            width_x = np.abs(x_95 - x_5)
            width_z = np.abs(z_95 - z_5)
            aspect_ratio = width_z / width_x if width_x != 0 else np.nan
        else:
            aspect_ratio = np.nan

        all_clusters.append({
            'site': row['label'],
            'WD': wd,
            'actin_points': row['actin_points'],
            'clathrin_points': row['clathrin_points'],
            'aspect_ratio': aspect_ratio
        })

# --- Filter to valid, low aspect ratio clusters (AR < 200) ---
subset = [c for c in all_clusters if not np.isnan(c['aspect_ratio']) and c['aspect_ratio'] > 3]

random.seed(2)
random_clusters = random.sample(subset, 1)

# --- Unit conversion: divide by 10, label as um ---
_unit_divisor = 10  # CHECK: nm -> um is /1000, not /10 -- confirm this matches your source units
_unit_label = 'μm'
_z_extra_divisor = 100  # additional Z-only scaling, applied on top of _unit_divisor

# --- Plot selected site ---
for cluster in random_clusters:
    site = cluster['site']
    wd_value = cluster['WD']  # pulled out to avoid quote conflict in the f-string below
    ar_value = cluster['aspect_ratio']

    fig, ax = plt.subplots(figsize=(30 / 25.4, 30 / 25.4))  # matches save_panel size below

    # Clathrin XZ -- absolute coordinates, converted units, no centering, Z further scaled by 100
    clathrin_pts = np.array(cluster['clathrin_points'])
    if len(clathrin_pts) > 0:
        clathrin_pts_conv = clathrin_pts / _unit_divisor
        z_clathrin_conv = (clathrin_pts_conv[:, 2] / _z_extra_divisor) if clathrin_pts_conv.shape[1] > 2 else np.zeros(len(clathrin_pts_conv))
        ax.scatter(
            clathrin_pts_conv[:, 0], z_clathrin_conv,
            c='deeppink', s=1, alpha=0.6, label='Clathrin'
        )

    # --- No manual xticks/yticks/tick_params overrides -- inherits global rcParams ---
    ax.set_ylim(-0.2, 0.2)
    mid_x = (clathrin_pts_conv[:, 0].min() + clathrin_pts_conv[:, 0].max()) / 2
    ax.set_xlim(mid_x - 0.05, mid_x + 0.05)
    ax.set_xlabel(f'X ({_unit_label})')
    ax.set_ylabel(f'Z ({_unit_label})')
    #ax.legend(loc='upper right', markerscale=3)
    #ax.set_aspect('equal')
    ax.set_title(f"Actin Positive (AR {ar_value:.2f})", fontsize=8, fontweight='bold')
    save_panel(fig, '2a_4', subdir='figure2', w_mm=30, h_mm=30)
    plt.tight_layout()
    plt.show()

In [ ]:
cmap = sns.color_palette("magma", 2)
control_color = cmap[0]  # dark purple
osmo_color = cmap[-1]    # bright yellow

# Prepare lists
wd_control = []
ar_control = []

wd_osmo = []
ar_osmo = []

for key, segment_dfs in filtered_all_segment_dfs.items():
    if 'control' in key.lower():
        target_wd = wd_control
        target_ar = ar_control
    elif 'osmo' in key.lower():
        target_wd = wd_osmo
        target_ar = ar_osmo
    else:
        continue

    for (gx, gy), df in segment_dfs.items():
        for _, row in df.iterrows():
            clathrin_pts = row['clathrin_points']
            if len(clathrin_pts) > 0:
                xs = clathrin_pts[:, 0]
                zs = clathrin_pts[:, 2] / 1000
                x_range = xs.max() - xs.min()
                z_range = zs.max() - zs.min()
                if x_range > 0:
                    aspect = z_range / x_range
                    target_ar.append(aspect)

                    # Handle WD arrays by taking mean
                    wd = row['wasserstein_distance']
                    if isinstance(wd, (list, np.ndarray)):
                        wd = np.mean(wd)
                    target_wd.append(wd)


In [ ]:


# --- Helper function: linear fit with CI and slope ---
def add_linear_fit_ci(ax, x, y, color, label):
    x = np.array(x)
    y = np.array(y)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 2:
        return

    # Fit linear model
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()
    r2 = model.rsquared
    slope = model.params[1]

    # Prediction for CI
    x_fit = np.linspace(np.min(x), np.max(x), 100)
    X_fit = sm.add_constant(x_fit)
    y_fit = model.predict(X_fit)
    pred = model.get_prediction(X_fit)
    ci = pred.conf_int(alpha=0.05)  # 95% CI

    # Plot fit line
    ax.plot(x_fit, y_fit, color=color, lw=2, label=f'{label} fit, R²={r2:.2f}')
    # Plot shaded confidence interval
    ax.fill_between(x_fit, ci[:,0], ci[:,1], color=color, alpha=0.2)

# --- Colors ---
cmap = sns.color_palette("magma", 2)
control_color = cmap[0]
osmo_color = cmap[-1]

# --- Create figure with 2 subplots ---
fig, axes = plt.subplots(1, 2, figsize=(14,8), sharey=True, sharex=True)
#plt.xlim(0,2000)

# --- Left plot: Filtered clusters (+) ---
ax = axes[0]
ax.scatter(ar_control, wd_control, color=control_color, alpha=0.5, s=10, label='Control +')
ax.scatter(ar_osmo, wd_osmo, color=osmo_color, alpha=0.5, s=10, label='Osmo +')
add_linear_fit_ci(ax, ar_control, wd_control, control_color, 'Control +')
add_linear_fit_ci(ax, ar_osmo, wd_osmo, osmo_color, 'Osmo +')
ax.set_xlabel('Clathrin Aspect Ratio (z/x)')
ax.set_ylabel('Wasserstein Distance (Rad)')
ax.set_title('Actin + Clusters')
ax.legend()


plt.tight_layout()
plt.show()


In [ ]:
def plot_binned_median(ar_list, wd_list, color, label, n_bins=10):
    df = pd.DataFrame({'AR': ar_list, 'WD': wd_list}).dropna()
    # Equal-count bins
    df['bin'] = pd.qcut(df['AR'], q=n_bins, duplicates='drop')
    # Compute median and IQR
    median = df.groupby('bin')['WD'].median()
    q25 = df.groupby('bin')['WD'].quantile(0.25)
    q75 = df.groupby('bin')['WD'].quantile(0.75)
    # Bin centers
    bin_centers = df.groupby('bin')['AR'].mean()
    # Plot
    plt.plot(bin_centers, median, color=color, label=label)
    plt.fill_between(bin_centers, q25, q75, color=color, alpha=0.3)
    plt.scatter(bin_centers, median, color=color, s=50)  # median points

# Select colors from either end of magma
cmap = sns.color_palette("magma", 2)
control_color = cmap[0]  # dark purple
osmo_color = cmap[-1]    # bright yellow

plt.figure(figsize=(8,6))
plot_binned_median(ar_control, wd_control, color=control_color, label='Control')
plot_binned_median(ar_osmo, wd_osmo, color=osmo_color, label='Osmo')

plt.xlabel('Clathrin Aspect Ratio (z/x)')
plt.ylabel('Actin Wasserstein Distance (Rad)')
#plt.title('WD vs Clathrin Aspect Ratio (Binned Median ± IQR)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def plot_binned_median(ar_list, wd_list, color, label, n_bins=15):
    df = pd.DataFrame({'AR': ar_list, 'WD': wd_list}).dropna()
    # Equal-count bins
    df['bin'] = pd.qcut(df['AR'], q=n_bins, duplicates='drop')
    # Compute median and IQR
    median = df.groupby('bin')['WD'].median()
    q25 = df.groupby('bin')['WD'].quantile(0.25)
    q75 = df.groupby('bin')['WD'].quantile(0.75)
    # Bin centers
    bin_centers = df.groupby('bin')['AR'].mean()
    # Plot
    plt.plot(bin_centers, median, color=color, label=label)
    plt.fill_between(bin_centers, q25, q75, color=color, alpha=0.3)
    plt.scatter(bin_centers, median, color=color, s=50)  # median points

# Select colors from either end of magma
cmap = sns.color_palette("magma", 2)
control_color = cmap[0]  # dark purple
osmo_color = cmap[-1]    # bright yellow

plt.figure(figsize=(8,6))
plot_binned_median(ar_control, wd_control, color=control_color, label='Control')
plot_binned_median(ar_osmo, wd_osmo, color=osmo_color, label='Osmo')

plt.xlabel('Clathrin Aspect Ratio (z/x)')
plt.ylabel('Actin Wasserstein Distance')
#plt.title('WD vs Clathrin Aspect Ratio (Binned Median ± IQR)')
plt.legend()
plt.tight_layout()
plt.show()

# Figure 3

In [ ]:
# Dictionary to store segment_dfs for each file
all_segment_dfs = {}

# Iterate over all loaded localization DataFrames
for key, df_localizations in all_localizations.items():
    print(f"Processing {key}...")
    # Split channels
    df_ch1 = df_localizations[df_localizations['c'] == 1]
    df_ch2 = df_localizations[df_localizations['c'] == 2]
    df_ch9 = df_localizations[df_localizations['c'] == 9]  

    if len(df_ch2) == 0 or len(df_ch1) == 0:
        print(f"Skipping {key}: missing channel 1 or 2")
        continue  # skip if any channel is empty

    # --- Parameters ---
    grid_size = 50  # nm
    r_actin = 1.5
    height = 800

    # Global origin
    xmin = min(df_ch1['xc'].min(), df_ch2['xc'].min())
    ymin = min(df_ch1['yc'].min(), df_ch2['yc'].min())

    # Determine grid ranges
    gx_max = int((max(df_ch1['xc'].max(), df_ch2['xc'].max()) - xmin) // grid_size) + 1
    gy_max = int((max(df_ch1['yc'].max(), df_ch2['yc'].max()) - ymin) // grid_size) + 1

    # Storage for segment DataFrames
    segment_dfs = {}

    for gx in range(gx_max):
        for gy in range(gy_max):
            # --- Extract segment points ---
            mask1 = ((df_ch1['xc'] - xmin)//grid_size == gx) & ((df_ch1['yc'] - ymin)//grid_size == gy)
            mask2 = ((df_ch2['xc'] - xmin)//grid_size == gx) & ((df_ch2['yc'] - ymin)//grid_size == gy)
            mask9 = ((df_ch9['xc'] - xmin)//grid_size == gx) & ((df_ch9['yc'] - ymin)//grid_size == gy)

            pts1_seg = df_ch1[['xc','yc']].values[mask1]
            pts2_seg = df_ch2[['xc','yc']].values[mask2]
            pts9_seg = df_ch9[['xc','yc','zc']].values[mask9]

            if len(pts2_seg) == 0:
                continue  # skip empty segments

            # --- Step 1: DBSCAN + cluster filtering ---
            clusters_prefilter, clusters_postfilter = dbscan_clathrin_segment(
                gx, gy, df_ch2, eps=0.2, min_samples=50, circular_threshold=5, r_close=2, w_clathrin=3
            )

            if len(clusters_postfilter) == 0:
                continue  # skip if no good clusters

            # --- Step 2: Assign actin and NWASP points ---
            clusters_with_actin = assign_actin_to_clusters(clusters_postfilter, df_ch1, r=r_actin, h=height)
            clusters_with_nwasp = assign_actin_to_clusters(clusters_postfilter, df_ch9, r=r_actin, h=height)

            # --- Step 3: Compute circular Wasserstein distance ---
            clusters_final_wd = compute_circular_wd(clusters_with_actin)
            clusters_final_wd_nwasp = compute_circular_wd(clusters_with_nwasp)
            #clusters_final_wd_clathrin = compute_circular_wd(clusters_postfilter)

            rows = []
            for c_actin, c_nwasp in zip(clusters_final_wd, clusters_final_wd_nwasp):
                rows.append({
                    'label': c_actin['label'],
                    'center': c_actin['center'],
                    'clathrin_points': c_actin['clathrin_points'],
                    'actin_points': c_actin['actin_points'],
                    'wasserstein_distance': c_actin['wasserstein_distance'],
                    'nwasp_points': c_nwasp['actin_points'],
                    'nwasp_wasserstein_distance': c_nwasp['wasserstein_distance'],
                })

            df_seg = pd.DataFrame(rows)
            segment_dfs[(gx, gy)] = df_seg

    # Save segment_dfs for this file
    all_segment_dfs[key] = segment_dfs

    if len(segment_dfs) == 0:
        print(f"No segments with clusters found for {key}.")
    else:
        print(f"Processed {key}: {len(segment_dfs)} segments.")


In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np
import pandas as pd

# Initialize dictionaries
filtered_all_segment_dfs = {}
removed_all_segment_dfs = {}
filtered_noiseless_segment_dfs = {}

# Helper to convert arrays to lists safely
def safe_list(x):
    return x.tolist() if isinstance(x, np.ndarray) else x

# Loop over all datasets
for key, segment_dfs in all_segment_dfs.items():
    filtered_segment_dfs = {}
    removed_segment_dfs = {}
    noiseless_segment_dfs = {}
    
    for (gx, gy), df_seg in segment_dfs.items():
        rows_filtered = []
        rows_removed = []
        rows_noiseless = []
        
        for _, c in df_seg.iterrows():
            actin_pts = np.array(c['actin_points'])
            clathrin_pts = np.array(c['clathrin_points'])
            nwasp_pts = np.array(c['nwasp_points']) if 'nwasp_points' in c else np.empty((0,3))
            
            # If no actin points at all, mark as removed
            if len(actin_pts) == 0:
                rows_removed.append({
                    'label': c['label'],
                    'center': safe_list(c['center']),
                    'clathrin_points': safe_list(clathrin_pts),
                    'actin_points': safe_list(actin_pts),
                    'nwasp_points': safe_list(nwasp_pts),
                    'wasserstein_distance': c['wasserstein_distance'],
                    'nwasp_wasserstein_distance': c.get('nwasp_wasserstein_distance', np.nan)
                })
                continue
            
            # --- DBSCAN on actin points (x,y) ---
            db = DBSCAN(eps=0.2, min_samples=10)
            labels = db.fit_predict(actin_pts[:, :2])
            
            # Kept clusters (any DBSCAN cluster found)
            if np.any(labels != -1):
                # Apply same logic for nWASP if needed
                keep_nwasp = len(nwasp_pts) > 0
                if keep_nwasp:
                    db_n = DBSCAN(eps=0.2, min_samples=20)
                    n_labels = db_n.fit_predict(nwasp_pts[:, :2])
                    keep_nwasp = np.any(n_labels != -1)
                
                if keep_nwasp or len(nwasp_pts) == 0:  # allow if no nwasp points
                    # Full filtered
                    rows_filtered.append({
                        'label': c['label'],
                        'center': safe_list(c['center']),
                        'clathrin_points': safe_list(clathrin_pts),
                        'actin_points': safe_list(actin_pts),
                        'nwasp_points': safe_list(nwasp_pts),
                        'wasserstein_distance': c['wasserstein_distance'],
                        'nwasp_wasserstein_distance': c.get('nwasp_wasserstein_distance', np.nan)
                    })
                    
                    # Noiseless: only points assigned to clusters
                    assigned_actin = actin_pts[labels != -1]
                    assigned_nwasp = nwasp_pts[n_labels != -1] if keep_nwasp else nwasp_pts
                    rows_noiseless.append({
                        'label': c['label'],
                        'center': safe_list(c['center']),
                        'clathrin_points': safe_list(clathrin_pts),
                        'actin_points': safe_list(assigned_actin),
                        'nwasp_points': safe_list(assigned_nwasp),
                        'wasserstein_distance': c['wasserstein_distance'],
                        'nwasp_wasserstein_distance': c.get('nwasp_wasserstein_distance', np.nan)
                    })
                else:
                    # nWASP filtered out → remove cluster
                    rows_removed.append({
                        'label': c['label'],
                        'center': safe_list(c['center']),
                        'clathrin_points': safe_list(clathrin_pts),
                        'actin_points': safe_list(actin_pts),
                        'nwasp_points': safe_list(nwasp_pts),
                        'wasserstein_distance': c['wasserstein_distance'],
                        'nwasp_wasserstein_distance': c.get('nwasp_wasserstein_distance', np.nan)
                    })
            else:
                # All actin points noise → removed
                rows_removed.append({
                    'label': c['label'],
                    'center': safe_list(c['center']),
                    'clathrin_points': safe_list(clathrin_pts),
                    'actin_points': safe_list(actin_pts),
                    'nwasp_points': safe_list(nwasp_pts),
                    'wasserstein_distance': c['wasserstein_distance'],
                    'nwasp_wasserstein_distance': c.get('nwasp_wasserstein_distance', np.nan)
                })
        
        # Save per-grid DataFrames if non-empty
        if len(rows_filtered) > 0:
            filtered_segment_dfs[(gx, gy)] = pd.DataFrame(rows_filtered)
        if len(rows_removed) > 0:
            removed_segment_dfs[(gx, gy)] = pd.DataFrame(rows_removed)
        if len(rows_noiseless) > 0:
            noiseless_segment_dfs[(gx, gy)] = pd.DataFrame(rows_noiseless)
    
    # Store in main dictionaries
    filtered_all_segment_dfs[key] = filtered_segment_dfs
    removed_all_segment_dfs[key] = removed_segment_dfs
    filtered_noiseless_segment_dfs[key] = noiseless_segment_dfs
    
    # Print summary
    n_kept = sum(len(df) for df in filtered_segment_dfs.values())
    n_removed = sum(len(df) for df in removed_segment_dfs.values())
    n_noiseless = sum(len(df) for df in noiseless_segment_dfs.values())
    print(f"Filtered {key}, kept {n_kept} clusters, removed {n_removed} clusters, noiseless {n_noiseless} clusters")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Collect Wasserstein distances ---
wasserstein_vals = []

for key, segment_dfs in all_segment_dfs.items():
    for (gx, gy), df in segment_dfs.items():
        vals = df['wasserstein_distance'].dropna().values
        wasserstein_vals.extend(vals)

wasserstein_vals = np.array(wasserstein_vals)

# --- Plot histogram using plt.hist ---
plt.figure(figsize=(8,6))
plt.hist(wasserstein_vals, bins=40, color='teal', edgecolor='black')
plt.xlabel("N-WASP Wasserstein Distance")
plt.ylabel("Count")
#plt.title("Distribution of NWASP Wasserstein Distances")
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()


In [ ]:
def compute_aspect_ratio_xz(points):
    """Compute aspect ratio (x/z) based on 5th–95th percentile spread."""
    if len(points) < 2:
        return np.nan
    pts = np.array(points)
    x5, x95 = np.percentile(pts[:, 0], [5, 95])
    z5, z95 = np.percentile(pts[:, 2], [5, 95])
    x_width = x95 - x5
    z_height = z95 - z5
    if z_height == 0:
        return np.nan
    return z_height / x_width # aspect ratio in XZ plane

# --- Collect data across all datasets ---
aspect_ratios = []
wasserstein_vals = []
dataset_labels = []

for key, segment_dfs in all_segment_dfs.items():
    for (gx, gy), df in segment_dfs.items():
        for _, row in df.iterrows():
            points = row.get('clathrin_points')
            wasser = row.get('wasserstein_distance', np.nan)

            if points is None or len(points) < 2 or np.isnan(wasser):
                continue

            aspect = compute_aspect_ratio_xz(points)
            if not np.isnan(aspect):
                aspect_ratios.append(aspect)
                wasserstein_vals.append(wasser)
                dataset_labels.append(key)


In [ ]:
# --- Flatten Wasserstein values if they are lists ---
wasserstein_vals_flat = []
aspect_ratios_flat = []

for ar, wd in zip(aspect_ratios, wasserstein_vals):
    # Take numeric value if list/array of length 1
    if isinstance(wd, (list, np.ndarray)) and len(wd) == 1:
        wd_val = wd[0]
    else:
        wd_val = wd

    if np.isscalar(ar) and np.isscalar(wd_val):
        if np.isfinite(ar) and np.isfinite(wd_val):
            aspect_ratios_flat.append(ar)
            wasserstein_vals_flat.append(wd_val)

# Convert to DataFrame
df_plot = pd.DataFrame({
    'AspectRatioXZ': aspect_ratios_flat,
    'Wasserstein': wasserstein_vals_flat
})
# --- Define bins ---
num_bins = 5
bins = np.linspace(df_plot['AspectRatioXZ'].min(), df_plot['AspectRatioXZ'].max(), num_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

medians = []
q25 = []
q75 = []

# --- Compute statistics per bin ---
for i in range(num_bins):
    mask = (df_plot['AspectRatioXZ'] >= bins[i]) & (df_plot['AspectRatioXZ'] < bins[i+1])
    bin_vals = df_plot.loc[mask, 'Wasserstein']
    if len(bin_vals) == 0:
        medians.append(np.nan)
        q25.append(np.nan)
        q75.append(np.nan)
    else:
        medians.append(np.median(bin_vals))
        q25.append(np.percentile(bin_vals, 25))
        q75.append(np.percentile(bin_vals, 75))

# --- Plot ---
plt.figure(figsize=(8,6))
plt.plot(bin_centers, medians, 'o-', color='teal', label='NWASP')
plt.fill_between(bin_centers, q25, q75, color='teal', alpha=0.3)

plt.xlabel("Clathrin Aspect Ratio (z/x)")
plt.ylabel("Wasserstein Distance")
#plt.title("NWASP Wasserstein Distance vs Clathrin Aspect Ratio")
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()


# Figure 4

In [ ]:
cmap = sns.color_palette("magma", 2)
control_color = cmap[0]  # dark purple
osmo_color = cmap[-1]    # bright yellow

# Prepare lists
wd_control = []
ar_control = []

wd_osmo = []
ar_osmo = []

for key, segment_dfs in filtered_all_segment_dfs.items():
    if 'control' in key.lower():
        target_wd = wd_control
        target_ar = ar_control
    elif 'osmo' in key.lower():
        target_wd = wd_osmo
        target_ar = ar_osmo
    else:
        continue

    for (gx, gy), df in segment_dfs.items():
        for _, row in df.iterrows():
            clathrin_pts = row['clathrin_points']
            if len(clathrin_pts) > 0:
                xs = clathrin_pts[:, 0]
                zs = clathrin_pts[:, 2] / 1000
                x_range = xs.max() - xs.min()
                z_range = zs.max() - zs.min()
                if x_range > 0:
                    aspect = z_range / x_range
                    target_ar.append(aspect)

                    # Handle WD arrays by taking mean
                    wd = row['wasserstein_distance']
                    if isinstance(wd, (list, np.ndarray)):
                        wd = np.mean(wd)
                    target_wd.append(wd)

In [ ]:
# --- Binned median + IQR plotting function -- no per-call styling,
# so it fully inherits font/tick/spine settings from the centralized rcParams
# block run earlier in the notebook. ---
def plot_binned_median(ax, ar_list, wd_list, color, label, n_bins=10):
    df = pd.DataFrame({'AR': ar_list, 'WD': wd_list}).dropna()
    # Equal-count bins
    df['bin'] = pd.qcut(df['AR'], q=n_bins, duplicates='drop')
    # Compute median and IQR
    median = df.groupby('bin')['WD'].median()
    q25 = df.groupby('bin')['WD'].quantile(0.25)
    q75 = df.groupby('bin')['WD'].quantile(0.75)
    # Bin centers
    bin_centers = df.groupby('bin')['AR'].mean()
    # Plot
    ax.plot(bin_centers, median, color=color, label=label)
    ax.fill_between(bin_centers, q25, q75, color=color, alpha=0.3)
    ax.scatter(bin_centers, median, color=color, s=20, alpha=0.35)  # median points


# --- Colors matching your earlier convention ---
cmap = sns.color_palette("magma", 2)
control_color = cmap[0]  # dark purple
osmo_color = cmap[-1]    # bright yellow

# --- Convert WD to Uniformity: 2*pi - WD ---
uniformity_control = [2 * np.pi - wd for wd in wd_control]
uniformity_osmo = [2 * np.pi - wd for wd in wd_osmo]

# --- Panel size: single-column square target, matching your other Figure 2 panels ---
w_mm, h_mm = 85, 65
fig, ax = plt.subplots(figsize=(w_mm / 25.4, h_mm / 25.4))

plot_binned_median(ax, ar_control, uniformity_control, color=control_color, label='Control')
plot_binned_median(ax, ar_osmo, uniformity_osmo, color=osmo_color, label='Osmotic shock')

ax.set_xlabel('Clathrin aspect ratio (z/x)')
ax.set_ylabel('Actin uniformity (rad)')
#ax.set_title('Uniformity vs Clathrin Aspect Ratio (Binned Median ± IQR)')  # titles stay out of the image file
ax.legend()

plt.tight_layout()
save_panel(fig, '4b', subdir='figure4', w_mm=w_mm, h_mm=h_mm)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from statannotations.Annotator import Annotator

# --- Collect WD values from filtered (Positive) and removed (Negative), split by control/osmo ---
def collect_wd(segment_dfs_dict):
    """segment_dfs_dict: e.g. filtered_all_segment_dfs or removed_all_segment_dfs"""
    control_wd, osmo_wd = [], []
    for dataset_key, seg_dfs in segment_dfs_dict.items():
        if 'control' in dataset_key.lower():
            target = control_wd
        elif 'osmo' in dataset_key.lower():
            target = osmo_wd
        else:
            continue

        for df_seg in seg_dfs.values():
            for _, row in df_seg.iterrows():
                wd = row['wasserstein_distance']
                if isinstance(wd, (list, np.ndarray)):
                    wd = float(np.squeeze(wd))
                if wd is None or (isinstance(wd, float) and np.isnan(wd)):
                    continue
                target.append(wd)
    return control_wd, osmo_wd

control_wd_pos, osmo_wd_pos = collect_wd(filtered_all_segment_dfs)   # Positive
control_wd_neg, osmo_wd_neg = collect_wd(removed_all_segment_dfs)    # Negative

# --- Convert WD (radians) to Uniformity: 2*pi - WD ---
uniformity_control_pos = [2 * np.pi - wd for wd in control_wd_pos]
uniformity_osmo_pos = [2 * np.pi - wd for wd in osmo_wd_pos]
uniformity_control_neg = [2 * np.pi - wd for wd in control_wd_neg]
uniformity_osmo_neg = [2 * np.pi - wd for wd in osmo_wd_neg]

# --- Prepare DataFrame ---
df_pos = pd.DataFrame({
    'WD': uniformity_control_pos + uniformity_osmo_pos,
    'Condition': ['Control'] * len(uniformity_control_pos) + ['Osmo'] * len(uniformity_osmo_pos),
    'Dataset': ['Positive'] * (len(uniformity_control_pos) + len(uniformity_osmo_pos))
})

df_neg = pd.DataFrame({
    'WD': uniformity_control_neg + uniformity_osmo_neg,
    'Condition': ['Control'] * len(uniformity_control_neg) + ['Osmo'] * len(uniformity_osmo_neg),
    'Dataset': ['Negative'] * (len(uniformity_control_neg) + len(uniformity_osmo_neg))
})

df_plot = pd.concat([df_pos, df_neg], ignore_index=True)

# --- Color palette -- magma ---
cmap = sns.color_palette("magma", 2)
palette_hue = {'Control': cmap[0], 'Osmo': cmap[1]}

order = ['Positive', 'Negative']
hue_order = ['Control', 'Osmo']

fig, ax = plt.subplots(figsize=(85 / 25.4, 50 / 25.4))

sns.boxplot(
    data=df_plot, x='Dataset', y='WD', order=order,
    hue='Condition', hue_order=hue_order, palette=palette_hue,
    showfliers=False, linewidth=0.6, width=0.6,
    boxprops=dict(alpha=0.35), ax=ax
)

ax.set_xlabel('Actin +/-')
ax.set_ylabel('Actin uniformity (rad)')

# --- Only the specified pairwise comparisons ---
pairs = [
    (('Positive', 'Control'), ('Positive', 'Osmo')),
    (('Positive', 'Control'), ('Negative', 'Control')),
    (('Positive', 'Osmo'), ('Negative', 'Osmo'))
]

# --- CI-label adapter: filters on BOTH Dataset and Condition columns ---
def _ci_label_2d(df, group_a, group_b, column):
    dataset_a, cond_a = group_a
    dataset_b, cond_b = group_b
    vals_a = df.loc[(df['Dataset'] == dataset_a) & (df['Condition'] == cond_a), column]
    vals_b = df.loc[(df['Dataset'] == dataset_b) & (df['Condition'] == cond_b), column]
    lo, hi = boot_pct_ci(vals_a, vals_b, seed=0)
    return f'95% CI [{lo:.1f}%, {hi:.1f}%]'

annot = Annotator(ax, pairs, data=df_plot, x='Dataset', y='WD', hue='Condition',
                   order=order, hue_order=hue_order)
annot.configure(loc='outside', fontsize=6, line_width=0.6, verbose=0)
annot.set_custom_annotations([_ci_label_2d(df_plot, a, b, 'WD') for a, b in pairs])
annot.annotate()

# --- Legend ---
handles, labels_ = ax.get_legend_handles_labels()
ax.legend(handles, hue_order, title='Condition', loc='lower center')

sns.despine()
plt.tight_layout()
save_panel(fig, '4c', subdir='figure4', w_mm=85, h_mm=65)
plt.show()

print(f"Positive - Control: {len(control_wd_pos)}, Osmo: {len(osmo_wd_pos)}")
print(f"Negative - Control: {len(control_wd_neg)}, Osmo: {len(osmo_wd_neg)}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns

# --- Helper function: linear fit (in log10(x)) with CI and slope, plotted on a log-x axis ---
def add_linear_fit_ci_logx(ax, x, y, color, label):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & (x > 0)  # log requires x > 0
    x = x[mask]
    y = y[mask]
    if len(x) < 2:
        return

    log_x = np.log10(x)

    # Fit linear model in log10(x) space
    X = sm.add_constant(log_x)
    model = sm.OLS(y, X).fit()
    r2 = model.rsquared

    # Prediction for CI, evaluated on a log-spaced grid so the line renders smoothly on a log axis
    x_fit = np.logspace(np.log10(x.min()), np.log10(x.max()), 100)
    log_x_fit = np.log10(x_fit)
    X_fit = sm.add_constant(log_x_fit)
    y_fit = model.predict(X_fit)
    pred = model.get_prediction(X_fit)
    ci = pred.conf_int(alpha=0.05)  # 95% CI

    ax.plot(x_fit, y_fit, color=color, lw=2, label=f'{label} fit, R²={r2:.2f}')
    ax.fill_between(x_fit, ci[:, 0], ci[:, 1], color=color, alpha=0.2)

# --- Collect WD and actin localization count, split by control/osmo ---
wd_control, n_actin_control = [], []
wd_osmo, n_actin_osmo = [], []

for dataset_key, seg_dfs in filtered_all_segment_dfs.items():
    if 'control' in dataset_key.lower():
        target_wd, target_n = wd_control, n_actin_control
    elif 'osmo' in dataset_key.lower():
        target_wd, target_n = wd_osmo, n_actin_osmo
    else:
        continue

    for df_seg in seg_dfs.values():
        for _, row in df_seg.iterrows():
            wd = row['wasserstein_distance']
            if isinstance(wd, (list, np.ndarray)):
                wd = float(np.squeeze(wd))
            if wd is None or (isinstance(wd, float) and np.isnan(wd)):
                continue

            actin_pts = np.array(row['actin_points'])
            n_actin = len(actin_pts)

            target_wd.append(wd)
            target_n.append(n_actin)

# --- Colors matching your earlier convention ---
cmap = sns.color_palette("magma", 2)
control_color = cmap[0]
osmo_color = cmap[-1]

# --- Panel size: matches your other Figure 2/4 panels; no local rc_context --
# relies entirely on the centralized rcParams block run earlier in the notebook ---
w_mm, h_mm = 85, 65
fig, ax = plt.subplots(figsize=(w_mm / 25.4, h_mm / 25.4))

ax.scatter(n_actin_control, wd_control, color=control_color, alpha=0.35, s=5, label='Control')
ax.scatter(n_actin_osmo, wd_osmo, color=osmo_color, alpha=0.35, s=5, label='Osmo')

add_linear_fit_ci_logx(ax, n_actin_control, wd_control, control_color, 'Control')
add_linear_fit_ci_logx(ax, n_actin_osmo, wd_osmo, osmo_color, 'Osmo')

ax.set_xscale('log')
ax.set_xlabel('Number of actin localizations')
ax.set_ylabel('Wasserstein distance (rad)')
ax.legend()

plt.tight_layout()
save_panel(fig, '4d', subdir='figure4', w_mm=w_mm, h_mm=h_mm)
plt.show()

print(f"Control: {len(wd_control)} clusters | Osmo: {len(wd_osmo)} clusters")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns
from matplotlib.ticker import LogLocator, NullFormatter

# --- Helper function: linear fit (in log10(x)) with CI and slope, plotted on a log-x axis ---
def add_linear_fit_ci_logx(ax, x, y, color, label):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & (x > 0)  # log requires x > 0
    x = x[mask]
    y = y[mask]
    if len(x) < 2:
        return

    log_x = np.log10(x)

    # Fit linear model in log10(x) space
    X = sm.add_constant(log_x)
    model = sm.OLS(y, X).fit()
    r2 = model.rsquared

    # Prediction for CI, evaluated on a log-spaced grid so the line renders smoothly on a log axis
    x_fit = np.logspace(np.log10(x.min()), np.log10(x.max()), 100)
    log_x_fit = np.log10(x_fit)
    X_fit = sm.add_constant(log_x_fit)
    y_fit = model.predict(X_fit)
    pred = model.get_prediction(X_fit)
    ci = pred.conf_int(alpha=0.05)  # 95% CI

    ax.plot(x_fit, y_fit, color=color, lw=2, label=f'{label} fit, R²={r2:.2f}')
    ax.fill_between(x_fit, ci[:, 0], ci[:, 1], color=color, alpha=0.2)

# --- Collect WD and actin localization count, split by control/osmo ---
wd_control, n_actin_control = [], []
wd_osmo, n_actin_osmo = [], []

for dataset_key, seg_dfs in filtered_all_segment_dfs.items():
    if 'control' in dataset_key.lower():
        target_wd, target_n = wd_control, n_actin_control
    elif 'osmo' in dataset_key.lower():
        target_wd, target_n = wd_osmo, n_actin_osmo
    else:
        continue

    for df_seg in seg_dfs.values():
        for _, row in df_seg.iterrows():
            wd = row['wasserstein_distance']
            if isinstance(wd, (list, np.ndarray)):
                wd = float(np.squeeze(wd))
            if wd is None or (isinstance(wd, float) and np.isnan(wd)):
                continue

            actin_pts = np.array(row['actin_points'])
            n_actin = len(actin_pts)

            target_wd.append(wd)
            target_n.append(n_actin)

# --- Convert WD (radians) to Uniformity: 2*pi - WD ---
uniformity_control = [2 * np.pi - wd for wd in wd_control]
uniformity_osmo = [2 * np.pi - wd for wd in wd_osmo]

# --- Colors matching your earlier convention ---
cmap = sns.color_palette("magma", 2)
control_color = cmap[0]
osmo_color = cmap[-1]

# --- Panel size: matches your other Figure 2/4 panels; no local rc_context --
# relies entirely on the centralized rcParams block run earlier in the notebook ---
w_mm, h_mm = 85, 65
fig, ax = plt.subplots(figsize=(w_mm / 25.4, h_mm / 25.4))

ax.scatter(n_actin_control, uniformity_control, color=control_color, alpha=0.35, s=5, label='Control')
ax.scatter(n_actin_osmo, uniformity_osmo, color=osmo_color, alpha=0.35, s=5, label='Osmo')

add_linear_fit_ci_logx(ax, n_actin_control, uniformity_control, control_color, 'Control')
add_linear_fit_ci_logx(ax, n_actin_osmo, uniformity_osmo, osmo_color, 'Osmo')

ax.set_xscale('log')

# --- Denser log-scale ticks: major ticks at every integer multiple within each decade
# (e.g. 100, 200, 300...900, 1000), not just at powers of ten. Tune `subs` if this is
# still too sparse or too crowded once you see the actual rendered panel. ---
ax.xaxis.set_major_locator(LogLocator(base=10.0, subs=np.arange(1, 10), numticks=12))
ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=12))
ax.xaxis.set_minor_formatter(NullFormatter())

ax.set_xlabel('Number of actin localizations')
ax.set_ylabel('Actin uniformity (rad)')
ax.legend()

plt.tight_layout()
save_panel(fig, '4d', subdir='figure4', w_mm=w_mm, h_mm=h_mm)
plt.show()

print(f"Control: {len(wd_control)} clusters | Osmo: {len(wd_osmo)} clusters")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# --- Binned median + IQR plotting function -- no per-call styling,
# so it fully inherits font/tick/spine settings from the centralized rcParams
# block run earlier in the notebook. ---
def plot_binned_median(ax, x_list, y_list, color, label, n_bins=10):
    df = pd.DataFrame({'X': x_list, 'Y': y_list}).dropna()
    # Equal-count bins
    df['bin'] = pd.qcut(df['X'], q=n_bins, duplicates='drop')
    # Compute median and IQR
    median = df.groupby('bin')['Y'].median()
    q25 = df.groupby('bin')['Y'].quantile(0.25)
    q75 = df.groupby('bin')['Y'].quantile(0.75)
    # Bin centers
    bin_centers = df.groupby('bin')['X'].mean()
    # Plot
    ax.plot(bin_centers, median, color=color, label=label)
    ax.fill_between(bin_centers, q25, q75, color=color, alpha=0.3)
    ax.scatter(bin_centers, median, color=color, s=20, alpha=0.35)  # median points

# --- Collect WD and actin localization count, split by control/osmo ---
wd_control, n_actin_control = [], []
wd_osmo, n_actin_osmo = [], []

for dataset_key, seg_dfs in filtered_all_segment_dfs.items():
    if 'control' in dataset_key.lower():
        target_wd, target_n = wd_control, n_actin_control
    elif 'osmo' in dataset_key.lower():
        target_wd, target_n = wd_osmo, n_actin_osmo
    else:
        continue

    for df_seg in seg_dfs.values():
        for _, row in df_seg.iterrows():
            wd = row['wasserstein_distance']
            if isinstance(wd, (list, np.ndarray)):
                wd = float(np.squeeze(wd))
            if wd is None or (isinstance(wd, float) and np.isnan(wd)):
                continue

            actin_pts = np.array(row['actin_points'])
            n_actin = len(actin_pts)

            target_wd.append(wd)
            target_n.append(n_actin)

# --- Convert WD (radians) to Uniformity: 2*pi - WD ---
uniformity_control = [2 * np.pi - wd for wd in wd_control]
uniformity_osmo = [2 * np.pi - wd for wd in wd_osmo]

# --- Colors matching your earlier convention ---
cmap = sns.color_palette("magma", 2)
control_color = cmap[0]
osmo_color = cmap[-1]

# --- Panel size: matches your other Figure 2/4 panels; no local rc_context --
# relies entirely on the centralized rcParams block run earlier in the notebook ---
w_mm, h_mm = 85, 65
fig, ax = plt.subplots(figsize=(w_mm / 25.4, h_mm / 25.4))

plot_binned_median(ax, n_actin_control, uniformity_control, color=control_color, label='Control')
plot_binned_median(ax, n_actin_osmo, uniformity_osmo, color=osmo_color, label='Shock')

ax.set_xlabel('Number of actin localizations')
ax.set_ylabel('Actin uniformity (rad)')
ax.legend(loc='lower right')

plt.tight_layout()
save_panel(fig, '4d', subdir='figure4', w_mm=w_mm, h_mm=h_mm)
plt.show()

print(f"Control: {len(wd_control)} clusters | Osmo: {len(wd_osmo)} clusters")